# 10-05 Agent 后训练

**核心概念**: 通过人类反馈进一步优化 Agent 行为，使其更符合人类偏好。

**本节目标**：RLHF、DPO、数据飞轮、Agent 自我进化

---

In [ ]:
# RLHF (Reinforcement Learning from Human Feedback) 概念演示
import random
random.seed(42)

print("""
RLHF 三阶段:

  阶段1: SFT (Supervised Fine-Tuning)
    - 用高质量 (instruction, response) 对微调基座模型
    - 让模型学会"怎么回答"的基本格式
  
  阶段2: 训练奖励模型 (Reward Model)
    - 人类标注员对多个回答排序: A > B > C
    - 训练 RM 学习人类偏好打分
  
  阶段3: PPO 强化学习优化
    - 用 RM 作为奖励信号
    - 用 PPO 算法优化 SFT 模型
    - KL 散度约束防止模型偏离太远
""")

# 模拟奖励模型
class SimpleRewardModel:
    """模拟奖励模型: 学习人类对广告文案的偏好"""
    
    def __init__(self):
        self.preferences = {
            "简洁": 2.0,     # 人类偏好简洁文案
            "有号召力": 1.5,  # 偏好有行动召唤
            "合规": 3.0,     # 强偏好合规内容
            "创意": 1.0,     # 适度创意
        }
        self.penalties = {
            "过长": -2.0,
            "极限词": -5.0,  # 强惩罚极限词
            "无关": -3.0,
        }
    
    def score(self, text: str) -> float:
        """给文案打分"""
        reward = 0.0
        # 正向奖励
        if len(text) <= 20:
            reward += self.preferences["简洁"]
        if any(w in text for w in ['立即', '快来', '限时', '体验']):
            reward += self.preferences["有号召力"]
        if not any(w in text for w in ['最', '第一', '绝对']):
            reward += self.preferences["合规"]
        # 负向惩罚
        if len(text) > 40:
            reward += self.penalties["过长"]
        if any(w in text for w in ['最', '第一', '绝对']):
            reward += self.penalties["极限词"]
        
        return reward

rm = SimpleRewardModel()

# 对比不同文案的奖励分数
candidates = [
    "大会员限时特惠 快来体验",           # 简洁+号召力+合规
    "这是全网最好的大会员服务，绝对超值",  # 极限词
    "大会员年卡现在购买有优惠活动正在进行中赶快来参与吧不要错过这个难得的机会", # 过长
    "限时福利 追番无广告",               # 简洁+合规
]

print("=== 奖励模型打分 ===")
scored = [(c, rm.score(c)) for c in candidates]
scored.sort(key=lambda x: x[1], reverse=True)
for text, score in scored:
    print(f"  [{score:+.1f}] {text}")

In [ ]:
# DPO (Direct Preference Optimization)
print("""
DPO vs RLHF:

  RLHF: SFT → 训练 RM → PPO 优化 (三阶段，复杂)
  DPO:  SFT → 直接用偏好数据优化 (两阶段，简单)

  DPO 核心公式:
    L_DPO = -E[log σ(β · (log π(y_w|x)/π_ref(y_w|x) - log π(y_l|x)/π_ref(y_l|x)))]
    
    其中:
    - y_w: 人类偏好的回答 (winner)
    - y_l: 人类不偏好的回答 (loser)  
    - π: 当前策略模型
    - π_ref: 参考模型 (SFT 模型)
    - β: 温度参数，控制偏离程度

  DPO 优点:
    ✅ 不需要单独训练 RM
    ✅ 训练更稳定 (无 RL 的不稳定性)
    ✅ 实现更简单
    ✅ 显存需求更低
""")

# 构造 DPO 偏好数据
dpo_dataset = [
    {
        "prompt": "为游戏皮肤写广告标题",
        "chosen": "沉浸游戏体验 限时畅玩",           # 人类偏好
        "rejected": "这是最好的游戏皮肤，绝对值得买",  # 人类不偏好
    },
    {
        "prompt": "为大会员写广告文案", 
        "chosen": "追番无广告 大会员畅享",
        "rejected": "大会员是第一名的视频会员服务产品，功能最全面",
    },
    {
        "prompt": "为编程课写推广标题",
        "chosen": "零基础学编程 名师带飞",
        "rejected": "我们的编程课程是全网最好的课程没有之一赶紧来报名",
    },
]

print("=== DPO 训练数据示例 ===")
for i, item in enumerate(dpo_dataset):
    print(f"\n样本 {i+1}:")
    print(f"  Prompt:   {item['prompt']}")
    print(f"  Chosen:   ✅ {item['chosen']}")
    print(f"  Rejected: ❌ {item['rejected']}")

print("""
\n构造 DPO 数据的方法:
  1. 人工标注: 标注员直接选择偏好 (质量高，成本高)
  2. 模型生成+人工筛选: LLM 生成多个，人工选最好的
  3. AI 反馈 (RLAIF): 用强模型 (GPT-4) 标注弱模型输出
  4. 隐式反馈: 用户点击/采纳 = chosen，忽略/修改 = rejected
""")

In [ ]:
# 数据飞轮: Agent 自我进化系统
class DataFlywheel:
    """数据飞轮: 收集反馈 → 筛选数据 → 微调 → 更好的模型"""
    
    def __init__(self):
        self.feedback_pool = []
        self.sft_data = []
        self.dpo_pairs = []
        self.model_version = 1
    
    def collect_feedback(self, query: str, response: str, feedback: str, score: float):
        """收集用户反馈"""
        self.feedback_pool.append({
            "query": query,
            "response": response,
            "feedback": feedback,  # accepted/modified/rejected
            "score": score,
            "model_version": self.model_version,
        })
    
    def mine_training_data(self, threshold: float = 0.7):
        """从反馈池挖掘训练数据"""
        # SFT 数据: 高分回答
        for item in self.feedback_pool:
            if item["score"] >= threshold and item["feedback"] == "accepted":
                self.sft_data.append({
                    "instruction": item["query"],
                    "output": item["response"],
                })
        
        # DPO 数据: 同 query 不同评分的配对
        from collections import defaultdict
        by_query = defaultdict(list)
        for item in self.feedback_pool:
            by_query[item["query"]].append(item)
        
        for query, items in by_query.items():
            if len(items) >= 2:
                items.sort(key=lambda x: x["score"], reverse=True)
                self.dpo_pairs.append({
                    "prompt": query,
                    "chosen": items[0]["response"],
                    "rejected": items[-1]["response"],
                })
        
        print(f"  挖掘 SFT 数据: {len(self.sft_data)} 条")
        print(f"  挖掘 DPO 数据: {len(self.dpo_pairs)} 对")
    
    def trigger_finetune(self):
        """触发微调（模拟）"""
        if len(self.sft_data) >= 3:  # 实际需要更多
            self.model_version += 1
            print(f"  🔄 触发微调 → 模型 v{self.model_version}")
            print(f"     SFT 数据: {len(self.sft_data)} 条")
            print(f"     DPO 数据: {len(self.dpo_pairs)} 对")
            self.feedback_pool.clear()
            self.sft_data.clear()
            self.dpo_pairs.clear()
            return True
        print(f"  数据不足，等待更多反馈 ({len(self.sft_data)}/3)")
        return False

# 模拟飞轮运转
flywheel = DataFlywheel()

# 模拟用户交互反馈
interactions = [
    ("写游戏皮肤广告", "沉浸游戏体验 限时畅玩", "accepted", 0.9),
    ("写游戏皮肤广告", "买游戏皮肤来这里看看", "rejected", 0.3),
    ("写大会员广告", "追番无广告 畅享大会员", "accepted", 0.85),
    ("写大会员广告", "大会员很好用快来买", "modified", 0.5),
    ("写美妆广告", "夏日防晒 温和守护", "accepted", 0.95),
    ("写美妆广告", "最好的防晒产品", "rejected", 0.2),
]

print("=== 数据飞轮运转 ===\n")
print("Phase 1: 收集反馈")
for q, r, fb, score in interactions:
    flywheel.collect_feedback(q, r, fb, score)
    print(f"  [{fb:>8}] ({score:.1f}) {q} → {r}")

print("\nPhase 2: 挖掘训练数据")
flywheel.mine_training_data()

print("\nPhase 3: 触发微调")
flywheel.trigger_finetune()

In [ ]:
# Agent 后训练完整 Pipeline
print("""
=== Agent 后训练完整 Pipeline ===

  ┌──────────────────────────────────────────────────┐
  │                  Agent 后训练                      │
  ├──────────────────────────────────────────────────┤
  │                                                    │
  │  1. 基座模型 (Qwen/LLaMA/...)                      │
  │       ↓                                           │
  │  2. SFT 微调 (领域数据 1K-10K条)                    │
  │       ↓                                           │
  │  3. 偏好对齐                                       │
  │       ├─ RLHF: RM + PPO (效果好，复杂)              │
  │       └─ DPO: 直接偏好优化 (简单，推荐)              │
  │       ↓                                           │
  │  4. 部署上线 + A/B 测试                             │
  │       ↓                                           │
  │  5. 数据飞轮                                       │
  │       收集反馈 → 筛选数据 → 重新微调 → 循环          │
  │                                                    │
  └──────────────────────────────────────────────────┘

=== B站广告 Agent 后训练实践 ===

  Step 1: SFT
    - 数据: 1000条广告主采纳的文案 (instruction → output)
    - 方法: LoRA 微调 Qwen-7B
    - 效果: 文案合规率 70% → 90%

  Step 2: DPO 对齐
    - 数据: 500对偏好数据 (adopted vs rejected)
    - 方法: DPO 微调 SFT 模型
    - 效果: 文案采纳率 60% → 78%

  Step 3: 数据飞轮
    - 每月收集 1000+ 条用户反馈
    - 每季度用新数据微调一次
    - 持续 A/B 测试跟踪 CTR 提升

  关键指标:
    - 文案合规率: >95% (硬指标)
    - 广告主采纳率: >75%
    - A/B 测试 CTR 提升: >8%
    - 模型更新周期: 每季度
""")

## 面试速记

| 问题 | 要点 |
|------|------|
| RLHF 三阶段 | SFT → 训练奖励模型(RM) → PPO优化；RM学人类偏好打分 |
| DPO vs RLHF | DPO 不需要 RM，直接用偏好数据优化；更简单稳定，推荐 |
| DPO 数据怎么构造 | 人工标注、模型生成+人工筛选、AI反馈(RLAIF)、隐式用户反馈 |
| 数据飞轮 | 用户反馈→筛选数据→微调→更好模型→更多反馈，正循环 |
| 如何评估后训练效果 | A/B测试(CTR/CVR)、人工评估(采纳率)、自动评估(合规率) |